# FluxAssist: AI-Drafted Variance Explanations for Month-End Close

**Proposal to Management** | Group: _[Team name / members]_ | Course: _[Course]_ | Date: _[Date]_

## 1. Bottom Line Up Front (BLUF)

**I propose FluxAssist, an internal tool that uses the OpenAI API to draft first-pass explanations for account variances ("flux analysis") during month-end close.** Staff accountants currently write these explanations by hand: they pull GL detail, compare it to last period, figure out what moved, and type a narrative for each account that crosses a threshold. FluxAssist does the drafting. The accountant reviews, edits, and approves.

**Why I should care:**
- Flux commentary is repetitive, deadline-driven work that lands in the most compressed week of the month.
- The accounting talent pipeline is tight, so hours freed from drafting are hours redeployed to judgment work (reconciliations, review, analysis).
- The tool never posts entries or approves anything. A human signs off on every explanation, so the control environment stays intact.

**Ask:** Approve a 60-day pilot on one entity's P&L accounts, with a small API budget and part-time support from one analyst who knows Python.

## 2. Business Problem

**Who experiences it:** Staff and senior accountants on the close team, plus the controller who reviews their work.

**What the work looks like today:**

| Step | Who | Pain point |
|---|---|---|
| Run trial balance comparison (current vs. prior month / budget) | Staff accountant | Fast, mostly automated |
| Flag accounts over threshold (e.g., >10% and >$25K) | Staff accountant | Fast, rule-based |
| Pull GL detail and vendor/journal memos for each flagged account | Staff accountant | Slow, manual, repetitive |
| Write a plain-English explanation of what drove the change | Staff accountant | Slowest step, quality varies by writer |
| Review and send back for rework | Controller | Rework loops when explanations are vague ("timing") |

**Why it's worth solving:**

1. **Labor is scarce and expensive.** The U.S. Bureau of Labor Statistics reports: *"About 124,200 openings for accountants and auditors are projected each year, on average, over the decade"* and *"The median annual wage for accountants and auditors was $81,680 in May 2024."* ([BLS Occupational Outlook Handbook](https://www.bls.gov/ooh/business-and-financial/accountants-and-auditors.htm)). Every hour of drafting I remove is an hour of a hard-to-hire professional's time returned.
2. **Explaining findings is core to the job, not a side task.** BLS notes that *"In addition to examining and preparing financial documents, accountants and auditors must explain their findings."* (same source). Narrative writing is a real, recurring workload.
3. **Close speed matters.** Late or vague flux commentary delays management reporting and pushes review work into evenings.

## 3. Proposed Solution

**FluxAssist** is a lightweight internal web app (or Excel add-in in a later phase) used by the close team.

**What it does:**
- Takes the list of flagged accounts and the supporting GL detail for each.
- Sends a structured, de-identified summary of that detail to the OpenAI API.
- Gets back a draft explanation for each account, the top transactions driving the change, a confidence level, and a list of open questions the accountant should confirm.
- Shows the draft side by side with the source data so the accountant can verify every number before approving.

**Who uses it:**

| User | How they use it |
|---|---|
| Staff accountant | Generates drafts, verifies against GL, edits, submits |
| Controller / reviewer | Reviews approved explanations, sees which ones were AI-drafted vs. hand-written |
| Finance systems analyst | Maintains thresholds, prompt templates, and logs |

**What it does not do:** post journal entries, change balances, approve reconciliations, or send anything outside the close team. It drafts text. Humans decide.

## 4. How It Works

### Input → OpenAI API → Output

```
┌─────────────────────────┐     ┌──────────────────────────────┐     ┌──────────────────────────────┐
│ INPUT (Python script)   │     │ OPENAI API                   │     │ OUTPUT                       │
│                         │     │                              │     │                              │
│ • Trial balance export  │ ──▶ │ • System prompt: "You are a  │ ──▶ │ • Draft explanation per acct │
│   (current vs. prior)   │     │   staff accountant writing   │     │ • Top 3 driver transactions  │
│ • Threshold rules       │     │   flux commentary. Use ONLY  │     │ • Confidence: high/med/low   │
│ • GL detail for flagged │     │   the data provided."        │     │ • Open questions to verify   │
│   accounts (vendor,     │     │ • Structured Outputs forces  │     │ • Saved to review log with   │
│   memo, amount, date)   │     │   a fixed JSON schema        │     │   reviewer name + timestamp  │
└─────────────────────────┘     └──────────────────────────────┘     └──────────────────────────────┘
```

### Step by step

1. **Python pulls the data.** A script reads the trial balance export (CSV/Excel via `pandas`), calculates dollar and percent change, and flags accounts over threshold. This step is deterministic math, no AI involved.
2. **Python builds the prompt.** For each flagged account, the script summarizes the GL detail: largest transactions, new vendors, one-time items, and memo text. Customer and employee names are replaced with IDs before anything leaves the company's network.
3. **The OpenAI API drafts the explanation.** The request uses **Structured Outputs**, which OpenAI describes as *"a feature that ensures the model will always generate responses that adhere to your supplied JSON Schema"* ([OpenAI API docs](https://developers.openai.com/api/docs/guides/structured-outputs)). That means every response comes back with the same fields, so the app can display and log them reliably.
4. **Python validates the response.** The script checks that every dollar amount the model cites exists in the GL detail it was given. Any mismatch flags the draft for manual rewrite.
5. **The accountant reviews and approves.** Nothing is final until a named human approves it.

### Example (illustrative data, not real)

**What I send (simplified):**
```
Account: 6420 - Software Subscriptions
Prior month: $48,200 | Current month: $71,900 | Change: +$23,700 (+49%)
Top transactions this month:
  - Vendor V-1182, $18,000, memo "Annual license renewal - analytics platform"
  - Vendor V-0931, $4,100, memo "Added 12 seats"
  - Vendor V-0455, $1,600, memo "Overage charge"
```

**What comes back (JSON schema enforced):**
```json
{
  "account": "6420 - Software Subscriptions",
  "draft_explanation": "Increase of $23.7K (+49%) driven primarily by an $18.0K annual license renewal for the analytics platform (V-1182) expensed this month, plus $4.1K for 12 added seats (V-0931).",
  "driver_transactions": ["V-1182 $18,000", "V-0931 $4,100", "V-0455 $1,600"],
  "confidence": "medium",
  "questions_for_preparer": [
    "Should the $18.0K annual renewal be recorded as a prepaid and amortized over 12 months instead of expensed?"
  ]
}
```

That last field is where the tool adds value beyond typing: it surfaces a possible accounting issue (prepaid vs. expense) for a human to judge.

## 5. Business Value

### Illustrative savings model

The numbers below are **planning assumptions**, not measured results. The pilot exists to replace them with real data.

| Input | Value | Basis |
|---|---|---|
| Accountants writing flux commentary | 4 | Assumption for a mid-size close team |
| Hours per accountant per month on flux drafting | 10 | Assumption; pilot will measure |
| Reduction in drafting time | 40% | Assumption; drafting drops, review remains |
| Hourly wage | ~$39.27 | BLS median $81,680 ÷ 2,080 hours |
| **Hours returned per month** | **16** | 4 × 10 × 40% |
| **Wage value per year** | **~$7,540** | 16 × 12 × $39.27 (wage only, before benefits/overhead) |

The dollar figure is modest on purpose. The bigger value is **where those hours go** and **how consistent the output is**:

| Value driver | Why it matters |
|---|---|
| Faster close | Drafts are ready in minutes, so review starts earlier in close week |
| Consistent quality | Every explanation names the drivers and amounts; fewer "timing" non-answers sent back for rework |
| Catches issues | The "questions for preparer" field can flag misclassifications (like the prepaid example) |
| Scales without headcount | Adding entities or accounts adds API cost, not hires |
| Audit trail | Every draft, edit, and approval is logged with who and when |

## 6. Considerations

### Resources needed

| Resource | Detail |
|---|---|
| OpenAI API account | Business API platform account with a monthly spend limit set |
| Developer time | One analyst comfortable with Python, `pandas`, and REST APIs, part-time for the pilot |
| Data access | Read-only export of trial balance and GL detail (no write access to the ERP) |
| Close team time | 2 to 3 accountants to test drafts and give feedback |
| Governance sign-off | IT security and controller approval before any real data is sent |

### Risks and how I manage them

| Risk | What could go wrong | Mitigation |
|---|---|---|
| **Confabulation (made-up facts)** | The model writes a plausible driver that isn't in the data. NIST's Generative AI Profile ([NIST AI 600-1](https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf)) lists confabulation among the risks specific to generative AI. | Model only sees the company's GL detail; Python cross-checks every cited amount; human approval required |
| **Data privacy** | Sensitive financial or personal data leaves the company | De-identify names before sending. OpenAI states for API business data: *"By default, we do not use your business data for training our models"* and that it *"may securely retain API inputs and outputs for up to 30 days"* ([OpenAI Enterprise Privacy](https://openai.com/enterprise-privacy/)). Request zero data retention if eligible. |
| **Internal controls / SOX** | Auditors question AI-written support | Tool drafts only; the human reviewer remains the control owner; full log of drafts vs. final text |
| **Over-reliance** | Staff stop thinking critically and rubber-stamp drafts | Track edit rates; spot-check approved drafts; keep "low confidence" drafts mandatory-rewrite |
| **Vendor dependency and cost** | Price or model changes break the workflow | Monthly spend cap; prompts and schema stored in the company's code so the model can be swapped |
| **Garbage in, garbage out** | Vague GL memos produce vague explanations | Pilot will surface which accounts have poor memo hygiene, which is useful on its own |

### Governance framework

I would align the pilot to the [NIST AI Risk Management Framework (AI 100-1)](https://doi.org/10.6028/NIST.AI.100-1), which NIST describes as a resource *"to help manage the many risks of AI and promote trustworthy and responsible development and use of AI systems."* Practically, that means documenting the use case (Map), tracking accuracy and edit rates (Measure), and keeping human approval and spend controls in place (Manage).

## 7. Pilot Plan and Success Metrics

| Phase | Timing | Activity |
|---|---|---|
| Build | Weeks 1–3 | Python data prep, prompt + JSON schema, simple review screen |
| Shadow run | Close #1 | Accountants write explanations as usual; FluxAssist drafts in parallel; compare |
| Live pilot | Close #2 | Accountants start from FluxAssist drafts; reviewer approves as normal |
| Decision | Week 9 | Go / adjust / stop, based on metrics below |

| Metric | Target to continue |
|---|---|
| Drafting time per account | At least 30% lower than shadow-run baseline |
| Drafts approved with minor or no edits | 60% or more |
| Drafts citing an amount not in the GL | 0 reach final approval |
| Reviewer rework requests | Lower than baseline |
| Monthly API cost | Within approved cap |

## 8. Connection to Python, APIs, and AI

| Concept from class | Where it shows up in FluxAssist |
|---|---|
| **Python** | `pandas` reads exports, calculates variances, applies thresholds, de-identifies data, validates responses |
| **APIs** | The OpenAI API is called over HTTPS with an API key; requests and JSON responses are handled in Python |
| **Structured data / JSON** | Structured Outputs returns a fixed schema the app can parse and log |
| **AI (LLMs)** | The model turns numbers and memo text into readable, consistent narrative |
| **Responsible AI** | Human-in-the-loop approval, de-identification, validation checks, NIST AI RMF alignment |